In [1]:
!pip install -q timm

In [2]:
import os
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.datasets import DatasetFolder
import timm
from timm.data import create_transform
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ---------------------- Configuration ----------------------
rp2k_train_path = "/kaggle/input/datasets/khyeh0719/rp2k-dataset/rp2k_dataset/all/train"
rp2k_test_path = "/kaggle/input/datasets/khyeh0719/rp2k-dataset/rp2k_dataset/all/test"

IMAGE_SIZE = 256
BATCH_SIZE = 128
NUM_EPOCHS = 15
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
NUM_WORKERS = 4
SEED = 42
CACHE_DIR = "/kaggle/working/rp2k_cache"
ALLOWED_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

os.makedirs(CACHE_DIR, exist_ok=True)

# ---------------------- Cached dataset class ----------------------
class CachedDataset(DatasetFolder):
    def __init__(self, root, samples, targets, classes, transform):
        self.root = root
        self.samples = samples
        self.targets = targets
        self.classes = classes
        self.class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        self.transform = transform
        self.loader = lambda x: Image.open(x).convert('RGB')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = self.loader(path)
        if self.transform:
            img = self.transform(img)
        return img, target

# ---------------------- Dataset building with cache ----------------------
def build_dataset_with_progress(root, transform, cache_name):
    """Builds a dataset from scratch, caching class order and samples."""
    cache_path = os.path.join(CACHE_DIR, cache_name)
    if os.path.exists(cache_path):
        print(f"Loading cached dataset from {cache_path}")
        with open(cache_path, 'rb') as f:
            samples, targets, classes = pickle.load(f)
        return CachedDataset(root, samples, targets, classes, transform)

    print(f"Scanning {root} for the first time...")
    classes = []
    class_to_idx = {}
    samples = []

    dirs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    for class_name in tqdm(dirs, desc="Processing classes"):
        class_path = os.path.join(root, class_name)
        if class_name not in class_to_idx:
            class_to_idx[class_name] = len(classes)
            classes.append(class_name)
        class_idx = class_to_idx[class_name]

        for fname in os.listdir(class_path):
            if fname.lower().endswith(ALLOWED_EXTS):
                samples.append((os.path.join(class_path, fname), class_idx))

    if not samples:
        raise RuntimeError(f"No images found in {root}. Check path and extensions.")

    targets = [s[1] for s in samples]
    print(f"Found {len(classes)} classes, {len(samples)} images")
    print(f"Caching to {cache_path}")
    with open(cache_path, 'wb') as f:
        pickle.dump((samples, targets, classes), f)

    return CachedDataset(root, samples, targets, classes, transform)

def build_validation_dataset(root, transform, train_classes, cache_name):
    """
    Builds validation dataset using the SAME class order as the training set.
    Classes not present in `train_classes` are skipped (with warning).
    """
    cache_path = os.path.join(CACHE_DIR, cache_name)
    if os.path.exists(cache_path):
        print(f"Loading cached validation dataset from {cache_path}")
        with open(cache_path, 'rb') as f:
            samples, targets, classes = pickle.load(f)
        return CachedDataset(root, samples, targets, classes, transform)

    print(f"Scanning {root} for validation images (using training class mapping)...")
    train_class_to_idx = {cls: idx for idx, cls in enumerate(train_classes)}

    samples = []
    targets = []
    skipped = 0

    for class_name in tqdm(os.listdir(root), desc="Validation classes"):
        class_path = os.path.join(root, class_name)
        if not os.path.isdir(class_path):
            continue
        if class_name not in train_class_to_idx:
            skipped += 1
            continue
        class_idx = train_class_to_idx[class_name]
        for fname in os.listdir(class_path):
            if fname.lower().endswith(ALLOWED_EXTS):
                samples.append((os.path.join(class_path, fname), class_idx))
                targets.append(class_idx)

    if skipped > 0:
        print(f"Warning: Skipped {skipped} class(es) not present in training set.")

    if not samples:
        raise RuntimeError(f"No valid images found in {root}. Check path and classes.")

    print(f"Using {len(train_classes)} classes, found {len(samples)} validation images")
    # Cache with the training class list to maintain consistency
    with open(cache_path, 'wb') as f:
        pickle.dump((samples, targets, train_classes), f)

    return CachedDataset(root, samples, targets, train_classes, transform)

# ---------------------- Transforms ----------------------
train_transform = create_transform(
    input_size=(3, IMAGE_SIZE, IMAGE_SIZE),
    is_training=True,
    use_prefetcher=False,
    no_aug=False,
    scale=(0.08, 1.0),
    ratio=(0.75, 1.333),
    hflip=0.5,
    vflip=0.0,
    color_jitter=0.4,
    auto_augment='rand-m12-mstd0.5',
    interpolation='bicubic',
    mean=IMAGENET_DEFAULT_MEAN,
    std=IMAGENET_DEFAULT_STD,
    re_prob=0.35,
    re_mode='pixel',
    re_count=1
)

valid_transform = create_transform(
    input_size=(3, IMAGE_SIZE, IMAGE_SIZE),
    is_training=False,
    interpolation='bicubic',
    mean=IMAGENET_DEFAULT_MEAN,
    std=IMAGENET_DEFAULT_STD,
)

# ---------------------- Load datasets ----------------------
print("\nLoading training dataset...")
train_dataset = build_dataset_with_progress(rp2k_train_path, train_transform, "train_cache.pkl")
print(f"Training dataset size: {len(train_dataset)}")
num_classes = len(train_dataset.classes)
print(f"Number of classes: {num_classes}")

print("\nLoading validation dataset...")
# IMPORTANT: use the training classes to enforce label consistency
valid_dataset = build_validation_dataset(rp2k_test_path, valid_transform,
                                         train_dataset.classes, "valid_cache.pkl")
print(f"Validation dataset size: {len(valid_dataset)}")

# ---------------------- DEBUG: check label ranges early ----------------------
print("\n[DEBUG] Validating label ranges before training...")
train_targets = np.array(train_dataset.targets)
val_targets = np.array(valid_dataset.targets)

print(f"Training labels - min: {train_targets.min()}, max: {train_targets.max()}")
print(f"Validation labels - min: {val_targets.min()}, max: {val_targets.max()}")
print(f"Model num_classes = {num_classes}")

if val_targets.max() >= num_classes or val_targets.min() < 0:
    raise ValueError(
        f"Validation set contains invalid labels! "
        f"Range = [{val_targets.min()}, {val_targets.max()}], "
        f"allowed = [0, {num_classes - 1}]."
    )
print("[DEBUG] Label sanity check passed. Proceeding to training.\n")
# ---------------------------------------------------------------------------

# ---------------------- WeightedRandomSampler ----------------------
class_counts = np.bincount(train_dataset.targets)
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
sample_weights = class_weights[train_dataset.targets]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# ---------------------- Model ----------------------
print("\nCreating model...")
model = timm.create_model('tf_efficientnetv2_s_in21ft1k', pretrained=True, num_classes=num_classes,     drop_path_rate=0.1)
model = model.to(device, memory_format=torch.channels_last)
model = torch.compile(model, mode="default")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, fused=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=5e-6)
scaler = torch.amp.GradScaler('cuda')

# ---------------------- Training ----------------------
best_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")

    # Training
    model.train()
    train_loss = 0.0
    all_preds = []
    all_labels = []
    train_bar = tqdm(train_loader, desc="Training")

    for images, labels in train_bar:
        images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        train_bar.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    history['train_loss'].append(avg_train_loss)
    history['train_acc'].append(train_acc)

    # Validation
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    val_bar = tqdm(valid_loader, desc="Validation")

    with torch.no_grad():
        for images, labels in val_bar:
            images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            val_bar.set_postfix(loss=loss.item())

    avg_val_loss = val_loss / len(valid_loader)
    val_acc = accuracy_score(all_labels, all_preds)
    history['val_loss'].append(avg_val_loss)
    history['val_acc'].append(val_acc)

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print(f"LR: {current_lr:.2e}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/best_efficientnetv2_s_rp2k.pth")
        print(f"✅ Saved best model with validation accuracy: {best_acc:.4f}")

print(f"\n{'='*50}")
print(f"Training completed. Best validation accuracy: {best_acc:.4f}")

# ---------------------- Save backbone for fine-tuning ----------------------
backbone = model
backbone.classifier = nn.Identity()
torch.save({
    'backbone_state_dict': backbone.state_dict(),
    'num_classes': num_classes,
    'image_size': IMAGE_SIZE,
    'model_name': 'tf_efficientnetv2_s_in21ft1k'
}, "/kaggle/working/rp2k_pretrained_backbone.pth")

print("\nPre-trained backbone saved to /kaggle/working/rp2k_pretrained_backbone.pth")
print("You can now load this backbone for metric learning fine-tuning.")

Using device: cuda

Loading training dataset...
Scanning /kaggle/input/datasets/khyeh0719/rp2k-dataset/rp2k_dataset/all/train for the first time...


Processing classes:   0%|          | 0/2384 [00:00<?, ?it/s]

Found 2384 classes, 344854 images
Caching to /kaggle/working/rp2k_cache/train_cache.pkl
Training dataset size: 344854
Number of classes: 2384

Loading validation dataset...
Scanning /kaggle/input/datasets/khyeh0719/rp2k-dataset/rp2k_dataset/all/test for validation images (using training class mapping)...


Validation classes:   0%|          | 0/2388 [00:00<?, ?it/s]

Using 2384 classes, found 39453 validation images
Validation dataset size: 39453

[DEBUG] Validating label ranges before training...
Training labels - min: 0, max: 2383
Validation labels - min: 0, max: 2383
Model num_classes = 2384
[DEBUG] Label sanity check passed. Proceeding to training.


Creating model...


...

Validation:   0%|          | 0/309 [00:00<?, ?it/s]

Train Loss: 0.8625, Train Acc: 0.9459
Val Loss: 0.9479, Val Acc: 0.9284
LR: 1.71e-05
✅ Saved best model with validation accuracy: 0.9284

Epoch 15/15


Training:   0%|          | 0/2695 [00:00<?, ?it/s]

Validation:   0%|          | 0/309 [00:00<?, ?it/s]

Train Loss: 0.8511, Train Acc: 0.9481
Val Loss: 0.9409, Val Acc: 0.9293
LR: 5.00e-04
✅ Saved best model with validation accuracy: 0.9293

Training completed. Best validation accuracy: 0.9293

Pre-trained backbone saved to /kaggle/working/rp2k_pretrained_backbone.pth
You can now load this backbone for metric learning fine-tuning.
